In [ ]:
!pip install uv regex
!uv pip install tensorflow transformers torch numpy pandas

In [ ]:
import transformers as tf
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import numpy as np
import pandas as pd
from qwen_model import Small_LLM_Model

model_name = "Qwen/Qwen3-0.6B"
prompts = "/content/function_calling_tests.json"
functions_definition = "/content/functions_definition.json"
output_file = "/content/output.json"

In [ ]:
from typing import Callable
import numpy as np
import json

class RegexMask:
  start = 0
  def __init__(self, model, regex):
    self.model = model
    self.regex = regex

    self.vocab_path = self.model.get_path_to_vocab_file()
    self.vocab_map = self.read_json(self.vocab_path)
    self.allowed_tokens = []

  def read_json(self, path_file):
    with open(path_file) as file:
      data = json.load(file)
    return data

  def __call__(self, input_ids, logits):
    self.start_index = len(input_ids[0]) - 1
    if RegexMask.start == 0:
      partial_output_str = ''
    else:
      partial_output_str = self.model.decode(
          input_ids[0, self.start_index]
      )

    allowed_tokens_ids = []
    for token_str, token_id in self.vocab_map.items():
      if self.regex.match(partial_output_str + token_str, partial = True):
        allowed_tokens_ids.append(token_id)

    if not allowed_tokens_ids:
      print("*" * 10, "at logits", "*" * 10)
      print(partial_output_str)
      return logits
    print("*" * 20)
    print(f"Allowed Tokens: {allowed_tokens_ids}")
    print("*" * 20)
    mask = np.full_like(logits, -np.inf)
    for token_id in allowed_tokens_ids:
      mask[token_id] = 0.0 # Changed from mask[:, token_id] = 0.0

    logits = logits + mask
    RegexMask.start += 1
    return logits

In [ ]:
from typing import Any
import regex

class Predictor:
    def __init__(self, model_name, file_definition, file_output) -> None:
        self.model = Small_LLM_Model(model_name)
        self.file_definition = file_definition
        self.file_output = file_output

        self.regex_format = r'^\{\s*"prompt"\s*:\s*"[^"]*"\s*,\s*"name"\s*:\s*"[^"]*"\s*,\s*"parameters"\s*:\s*(?P<args>\{[^{}]*\})\s*\}$'
        self.regex = regex.compile(self.regex_format)
        self.expected_output = """
        {
        "prompt": "string",
        "name": "string",
        "parameters": "dictionary"
        }"""

        self.rules = """
        - Always include "name", "parameters" and "prompt" as keys.
        - and "name" must be in function definition.
        - Do NOT add extra fields.
        - Do NOT remove required fields.
        - Use null if a value is missing
        - Numbers must be numeric (not strings)
        - Argument types must match the function definition (number, string, boolean, etc.)
        - Use double quotes for all keys and strings
        - No trailing commas
        - Output must start with '{' and end with '}'
        """
        self.result = ""

    def read_file(self, file_path) -> Any:
      if not file_path:
        pass

      with open(file_path, 'r') as file:
        data = file.read()
      return data

    def write_file(self, file_path, data=None) -> None:
      if not file_path:
        pass

      if not data:
        pass

      if not file_path.endswith('.json'):
        pass

      try:
        with (file_path, 'w') as file:
          file.write(data)
      except Exception as e:
        pass

    def encode(self, prompt) -> list:
      return self.model.encode(prompt)

    def decode(self, ids: list) -> str:
      return self.model.decode(ids)

    def softmax(self, data: list) -> list:
      exp_value = np.exp(data)
      exp_sum = np.sum(exp_value)
      return exp_value / exp_sum

    def predict_next_token(self, prompt) -> float:
      self.next_token = 0
      ids = self.model.encode(prompt)
      logits = self.model.get_logits_from_input_ids(ids.tolist()[0])

      mask = RegexMask(self.model, self.regex)
      logits = mask(ids, logits)

      logits = self.softmax(logits)
      self.next_token = np.argmax(logits)
      return self.next_token

    def decode_next_token(self, next_token) -> str:
      return self.model.decode(next_token)

    def generate_text(self, prompt) -> None:
      text = self.init_prompt(prompt)

      adding_to_string = False
      self.output_text: str = ""

      while True:
        next_token = self.predict_next_token(text)
        next_word = self.decode_next_token(next_token)
        text += next_word
        print(f"text: {text}")

        if adding_to_string:
          self.output_text += next_word

        if '{' in next_word and not adding_to_string:
          self.output_text += next_word
          adding_to_string = True

        if self.output_text.endswith('}'):
          try:
            json.loads(self.output_text)
            break
          except json.JSONDecodeError:
            pass

    def save_output(self) -> None:
      self.write_file(self.output_text)

    def get_output(self) -> str:
      return self.output_text

    def init_prompt(self, prompt):
      llm_prompt = f"""
      You are a function-calling JSON generator.

Your task:
- Read the input carefully
- Select the correct function
- Extract the required parameters
- Return ONLY valid JSON

Do NOT output:
- explanations
- text
- comments
- markdown
- code blocks

If you output anything other than JSON, it is incorrect.

Function definition:
{self.read_file(self.file_definition)}

Input:
{prompt}

Output schema (MUST match exactly):
{self.expected_output}

Rules:
{self.rules}

Example:

prompt:
What is the sum of 2 and 3?

Output:
{{
  "prompt": "What is the sum of 2 and 3?",
  "name": "fn_add_numbers",
  "parameters": {{
    "a": 2,
    "b": 3
  }}
}}

Now process this input:

{{your_input_here}}
      """
      return llm_prompt


In [ ]:
model = Predictor(model_name, functions_definition, output_file)

In [ ]:
import json
with open(prompts, 'r') as file:
  data = json.load(file)

model.generate_text(data[0]['prompt'])

Streaming output truncated to the last 5000 lines.

********** at logits **********
,

text: 
      You are a function-calling JSON generator.

Your task:
- Read the input carefully
- Select the correct function
- Extract the required parameters
- Return ONLY valid JSON

Do NOT output:
- explanations
- text
- comments
- markdown
- code blocks

If you output anything other than JSON, it is incorrect.

Function definition:
[
  {
    "name": "fn_add_numbers",
    "description": "Add two numbers together and return their sum.",
    "parameters": {
      "a": {
        "type": "number"
      },
      "b": {
        "type": "number"
      }
    },
    "returns": {
      "type": "number"
    }
  },
  {
    "name": "fn_greet",
    "description": "Generate a greeting message for a person by name.",
    "parameters": {
      "name": {
        "type": "string"
      }
    },
    "returns": {
      "type": "string"
    }
  },
  {
    "name": "fn_reverse_string",
    "description": "Reverse a strin

In [ ]:
print(model.get_output())

In [ ]:
import json
import pandas as pd
import numpy as np
import regex

vocab = model.model.get_path_to_vocab_file()
with open(vocab, 'r') as file:
  vocab = json.load(file)

allowed_tokens = []
test_str = ' '
regex_format = r'^\{\s*"prompt"\s*:\s*"[^"]*"\s*,\s*"name"\s*:\s*"[^"]*"\s*,\s*"parameters"\s*:\s*(?P<args>\{[^{}]*\})\s*/}$'
regex_model = regex.compile(regex_format)
for token_str, token_id in vocab.items():
  if regex_model.match(test_str + token_str, partial = True):
    allowed_tokens.append(token_id)
    test_str += token_str


In [ ]:
for token in allowed_tokens:
  print(f"token: '{model.decode(token)}', id: {token}")
  print(test_str)

In [ ]:
test = ' '
res = regex_model.match(test, partial=True)
res

In [ ]:
print(allowed_tokens)

In [ ]:
text = data[1]['prompt']
text

In [ ]:
ids = model.encode(text)
ids

In [ ]:
ids[0].tolist()

In [ ]:
ids[0, -1]